# Aggregate evaluation results

Reads the JSON files produced by `eval_sb3.py --json-out` and reports the success rate as mean +/- std over the seeds, plus a comparison plot.

In [ ]:
import glob, json, os
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = "results"

rows = []
for fp in glob.glob(os.path.join(RESULTS_DIR, "*.json")):
    with open(fp) as f:
        rows.append(json.load(f))
print(f"loaded {len(rows)} result files from {RESULTS_DIR}/")


## Aggregate over seeds

In [ ]:
groups = defaultdict(list)
for r in rows:
    if r.get("success_rate") is not None:
        groups[(r["algo"], r.get("strategy", "none"), r["env_type"])].append(r["success_rate"])

agg = {k: (float(np.mean(v)), float(np.std(v)), len(v)) for k, v in groups.items()}

print(f"{'algo':5} {'strategy':9} {'env':7} {'success':>8} {'std':>7} {'n':>3}")
for (algo, strat, env), (mean, std, n) in sorted(agg.items()):
    print(f"{algo:5} {strat:9} {env:7} {mean:8.3f} {std:7.3f} {n:>3}")


## Comparison plot: randomization strategy vs transfer

In [ ]:
strategies = ["none", "udr", "adr"]
envs = ["source", "target"]

for algo in sorted({k[0] for k in agg}):
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(strategies))
    width = 0.35
    for i, env in enumerate(envs):
        means = [agg.get((algo, s, env), (0, 0, 0))[0] for s in strategies]
        errs = [agg.get((algo, s, env), (0, 0, 0))[1] for s in strategies]
        ax.bar(x + (i - 0.5) * width, means, width, yerr=errs, capsize=4, label=env)
    ax.set_xticks(x)
    ax.set_xticklabels(strategies)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Success rate")
    ax.set_title(f"{algo.upper()} - randomization vs transfer")
    ax.legend()
    os.makedirs("report/imgs", exist_ok=True)
    out = f"report/imgs/transfer_comparison_{algo}.png"
    fig.tight_layout()
    fig.savefig(out, dpi=200)
    plt.show()
    print("saved", out)
